### JSON Parsing and Processing

In [26]:
import json
import os
os.makedirs("data/json_files",exist_ok=True)

In [27]:
json_data={
  "company": "TechCorp",
  "employees": [
    {
      "id": 1,
      "name": "John Doe",
      "role": "Software Engineer",
      "skills": [
        "Python",
        "JavaScript",
        "React"
      ],
      "projects": [
        {
          "name": "RAG System",
          "status": "In Progress"
        },
        {
          "name": "Data Pipeline",
          "status": "Completed"
        }
      ]
    },
    {
      "id": 2,
      "name": "Jane Smith",
      "role": "Machine Learning Engineer",
      "skills": [
        "Python",
        "PyTorch",
        "Hugging Face"
      ],
      "projects": [
        {
          "name": "Fine-Tuning LLM",
          "status": "In Progress"
        },
        {
          "name": "Semantic Search",
          "status": "Completed"
        }
      ]
    },
    {
      "id": 3,
      "name": "Michael Brown",
      "role": "DevOps Engineer",
      "skills": [
        "Docker",
        "Kubernetes",
        "AWS"
      ],
      "projects": [
        {
          "name": "CI/CD Pipeline",
          "status": "Completed"
        },
        {
          "name": "Cloud Migration",
          "status": "In Progress"
        }
      ]
    },
    {
      "id": 4,
      "name": "Emily Davis",
      "role": "Frontend Developer",
      "skills": [
        "TypeScript",
        "Next.js",
        "Tailwind CSS"
      ],
      "projects": [
        {
          "name": "Admin Dashboard",
          "status": "Completed"
        },
        {
          "name": "Analytics Portal",
          "status": "In Progress"
        }
      ]
    }
  ]
}

In [28]:
json_data

{'company': 'TechCorp',
 'employees': [{'id': 1,
   'name': 'John Doe',
   'role': 'Software Engineer',
   'skills': ['Python', 'JavaScript', 'React'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Jane Smith',
   'role': 'Machine Learning Engineer',
   'skills': ['Python', 'PyTorch', 'Hugging Face'],
   'projects': [{'name': 'Fine-Tuning LLM', 'status': 'In Progress'},
    {'name': 'Semantic Search', 'status': 'Completed'}]},
  {'id': 3,
   'name': 'Michael Brown',
   'role': 'DevOps Engineer',
   'skills': ['Docker', 'Kubernetes', 'AWS'],
   'projects': [{'name': 'CI/CD Pipeline', 'status': 'Completed'},
    {'name': 'Cloud Migration', 'status': 'In Progress'}]},
  {'id': 4,
   'name': 'Emily Davis',
   'role': 'Frontend Developer',
   'skills': ['TypeScript', 'Next.js', 'Tailwind CSS'],
   'projects': [{'name': 'Admin Dashboard', 'status': 'Completed'},
    {'name': 'Analytics Portal',

In [29]:
with open("data/json_files/company_data.json","w") as f:
    json.dump(json_data,f,indent=2)

In [30]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]

with open('data/json_files/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

### Json Processing Strategies

In [32]:
from langchain_community.document_loaders import JSONLoader
import json

In [35]:
# Method-1 JsonLoader with JQ_Schema
print("JSONLoader - Extract Specific fields")
# Extraxt employee information
employee_loader=JSONLoader(
    file_path="data/json_files/company_data.json",
    jq_schema=".employees[]", #jq query to extract each employee
    text_content=False #get fulll JSON objects
    )

employee_docs=employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First Employee: {employee_docs[0].page_content[:200]}....")
print(employee_docs)

JSONLoader - Extract Specific fields
Loaded 4 employee documents
First Employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status"....
[Document(metadata={'source': 'D:\\LangChain by Krish Naik\\0-DataingestParsing\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'), Document(metadata={'source': 'D:\\LangChain by Krish Naik\\0-DataingestParsing\\data\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Machine Learning Engineer", "skills": ["Python", "PyTorch", "Hugging Face"], "projects": [{"name": "Fine-Tuning LLM", "status": "In Progress"}, {"name": "Sema

In [39]:
# Method-2 Custom Json Processing for complex structure
from importlib import metadata
from typing import List
from langchain_core.documents import Document
print("\n Custom JSON Processing")

def process_json_intelligent(filepath:str)-> List[Document]:
    """Processing JSON with intelligent flattening and context preservation"""
    with open(filepath,'r') as f:
        data=json.load(f)

    documents=[]

    #Strategy 1: Create dosuments for each employee with full context
    for emp in data.get('employees',[]):
        content=f"""Employee Profile:
        Name:{emp['name']}
        Role:{emp['role']}
        Skills:{'. '.join(emp['skills'])}
        Projects:"""
        for proj in emp.get("projects",[]):
            content+=f"\n-{proj['name']}(Satus: {proj['status']})"

        doc=Document(
            page_content=content,
            metadata={
                'source':filepath,
                'data_type':'employee_profile',
                'employee_id':emp['id'],
                "employee_name":emp["name"],
                "role":emp["role"]
            }
        )
        documents.append(doc)
    return documents




 Custom JSON Processing


In [40]:
process_json_intelligent("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='Employee Profile:\n        Name:John Doe\n        Role:Software Engineer\n        Skills:Python. JavaScript. React\n        Projects:\n-RAG System(Satus: In Progress)\n-Data Pipeline(Satus: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Machine Learning Engineer'}, page_content='Employee Profile:\n        Name:Jane Smith\n        Role:Machine Learning Engineer\n        Skills:Python. PyTorch. Hugging Face\n        Projects:\n-Fine-Tuning LLM(Satus: In Progress)\n-Semantic Search(Satus: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 3, 'employee_name': 'Michael Brown', 'role': 'DevOps Engineer'},